# Building an Auto Insurance Quote Multi-Agent System

This tutorial will walk you through building a multi-agent system for processing car insurance quotes using LangGraph for the backend agents and React with Cloudscape for the frontend UI.

## Overview

We'll build a system with six specialized agents:
1. **Customer Information Agent**: Retrieves and analyzes customer information
2. **Vehicle Information Agent**: Retrieves and analyzes vehicle information
3. **Risk Assessment Agent**: Assesses risk based on customer and vehicle information
4. **Coverage Determination Agent**: Determines appropriate coverage based on customer request and risk assessment
5. **Pricing Agent**: Calculates the final price based on all factors
6. **Quote Generation Agent**: Generates the final quote with all details

## Part 1: Project Setup

Let's start by setting up our project structure.

In [ ]:
# Create project directories
!mkdir -p backend
!mkdir -p frontend/public
!mkdir -p frontend/src/components
!mkdir -p frontend/src/pages
!mkdir -p data

## Part 2: Setting Up the Backend

First, let's create our backend files. We'll start with the requirements.txt file.

In [ ]:
%%writefile backend/requirements.txt
fastapi==0.104.1
uvicorn==0.23.2
langchain==0.0.335
langgraph==0.0.20
langchain-aws==0.0.2
boto3==1.28.64
python-dotenv==1.0.0
pydantic==2.4.2

Next, let's create a .env.example file for the backend:

In [ ]:
%%writefile backend/.env.example
AWS_REGION=us-west-2
BEDROCK_MODEL_ID=anthropic.claude-3-sonnet-20240229-v1:0

## Part 3: Creating Mock Data

Let's create some mock data files for our application.

In [ ]:
%%writefile data/customers.json
[
  {
    "id": "cust-001",
    "name": "John Smith",
    "age": 35,
    "address": "123 Main St, Seattle, WA 98101",
    "drivingHistory": {
      "licenseNumber": "WA12345678",
      "yearsLicensed": 17,
      "accidents": 0,
      "violations": 1
    },
    "creditScore": 720
  },
  {
    "id": "cust-002",
    "name": "Sarah Johnson",
    "age": 28,
    "address": "456 Pine Ave, Portland, OR 97205",
    "drivingHistory": {
      "licenseNumber": "OR87654321",
      "yearsLicensed": 10,
      "accidents": 1,
      "violations": 0
    },
    "creditScore": 680
  },
  {
    "id": "cust-003",
    "name": "Michael Chen",
    "age": 42,
    "address": "789 Oak Blvd, San Francisco, CA 94107",
    "drivingHistory": {
      "licenseNumber": "CA98765432",
      "yearsLicensed": 24,
      "accidents": 0,
      "violations": 0
    },
    "creditScore": 790
  }
]

In [ ]:
%%writefile data/vehicles.json
[
  {
    "vin": "1HGCM82633A123456",
    "make": "Honda",
    "model": "Accord",
    "year": 2020,
    "value": 25000,
    "safetyRating": 4.5,
    "theftRating": 2.1
  },
  {
    "vin": "5YJSA1E47JF123456",
    "make": "Tesla",
    "model": "Model S",
    "year": 2022,
    "value": 85000,
    "safetyRating": 5.0,
    "theftRating": 3.2
  },
  {
    "vin": "1FTEW1E85JKC98765",
    "make": "Ford",
    "model": "F-150",
    "year": 2019,
    "value": 32000,
    "safetyRating": 4.0,
    "theftRating": 3.8
  }
]

In [ ]:
%%writefile data/products.json
[
  {
    "id": "basic",
    "name": "Basic Coverage",
    "description": "Minimum required coverage for legal compliance",
    "coverages": [
      {
        "type": "liability",
        "bodily_injury": "25000/50000",
        "property_damage": "25000"
      }
    ],
    "base_premium": 600
  },
  {
    "id": "standard",
    "name": "Standard Coverage",
    "description": "Balanced protection for most drivers",
    "coverages": [
      {
        "type": "liability",
        "bodily_injury": "50000/100000",
        "property_damage": "50000"
      },
      {
        "type": "collision",
        "deductible": 500
      },
      {
        "type": "comprehensive",
        "deductible": 500
      }
    ],
    "base_premium": 1200
  },
  {
    "id": "premium",
    "name": "Premium Coverage",
    "description": "Maximum protection for complete peace of mind",
    "coverages": [
      {
        "type": "liability",
        "bodily_injury": "100000/300000",
        "property_damage": "100000"
      },
      {
        "type": "collision",
        "deductible": 250
      },
      {
        "type": "comprehensive",
        "deductible": 250
      },
      {
        "type": "uninsured_motorist",
        "coverage": "100000/300000"
      },
      {
        "type": "roadside_assistance"
      },
      {
        "type": "rental_reimbursement",
        "daily_limit": 50,
        "max_days": 30
      }
    ],
    "base_premium": 1800
  }
]

In [ ]:
%%writefile data/pricing_rules.json
{
  "age_factors": {
    "under_25": 1.5,
    "25_to_60": 1.0,
    "over_60": 1.2
  },
  "driving_history_factors": {
    "accident_multiplier": 1.25,
    "violation_multiplier": 1.15,
    "clean_record_discount": 0.9
  },
  "credit_score_factors": {
    "excellent": 0.85,
    "good": 0.95,
    "fair": 1.1,
    "poor": 1.3
  },
  "vehicle_factors": {
    "value_tiers": {
      "under_20k": 0.9,
      "20k_to_40k": 1.0,
      "40k_to_70k": 1.2,
      "over_70k": 1.5
    },
    "safety_rating_discount": 0.05,
    "high_theft_surcharge": 0.1
  },
  "location_factors": {
    "urban": 1.2,
    "suburban": 1.0,
    "rural": 0.9
  }
}

## Part 4: Visualizing the Agent System

Before we implement our agents, let's visualize the structure of our multi-agent system using Mermaid diagrams. This will help us understand the flow of information between agents.

In [ ]:
# Install mermaid-magic for Jupyter
!pip install mermaid-magic

In [ ]:
%load_ext mermaid_magic

In [ ]:
%%mermaid
graph TD
    A[User Request] --> B[Customer Information Agent]
    A --> C[Vehicle Information Agent]
    B --> D[Risk Assessment Agent]
    C --> D
    D --> E[Coverage Determination Agent]
    E --> F[Pricing Agent]
    F --> G[Quote Generation Agent]
    G --> H[Final Quote]
    
    classDef agent fill:#f9f,stroke:#333,stroke-width:2px;
    classDef input fill:#bbf,stroke:#333,stroke-width:1px;
    classDef output fill:#bfb,stroke:#333,stroke-width:1px;
    
    class B,C,D,E,F,G agent;
    class A input;
    class H output;

## Part 5: Implementing the Agent System

Now let's implement our multi-agent system using LangGraph. We'll start by defining the state schema and agent nodes.

In [ ]:
import json
from typing import Dict, Optional, TypedDict

# Removed unused imports
# from langchain_aws import BedrockChat
# from langchain.prompts import ChatPromptTemplate
# from langgraph.graph import StateGraph, END

# Load mock data
with open('data/customers.json', 'r') as f:
    customers = json.load(f)
    
with open('data/vehicles.json', 'r') as f:
    vehicles = json.load(f)
    
with open('data/products.json', 'r') as f:
    products = json.load(f)
    
with open('data/pricing_rules.json', 'r') as f:
    pricing_rules = json.load(f)

### Define the State Schema

First, let's define the schema for our agent system state:

In [ ]:
class QuoteState(TypedDict):
    customer_id: str
    vehicle_info: Optional[Dict]
    customer_info: Optional[Dict]
    risk_assessment: Optional[Dict]
    coverage_recommendation: Optional[Dict]
    price_calculation: Optional[Dict]
    final_quote: Optional[Dict]

## Part 6: Setting Up FastAPI

In the next part, we'll implement the FastAPI application to serve our agent system. Let's create the basic structure for our API.

In [ ]:
%%writefile backend/app.py
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import Dict, Optional
import json
import os

# This will be replaced with the actual agent system import
# from agent_system import process_quote_request

app = FastAPI(title="Auto Insurance Quote API")

# Add CORS middleware
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],  # For development; restrict in production
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# Load mock data
def load_mock_data():
    data_dir = os.path.join(os.path.dirname(os.path.dirname(__file__)), "data")
    
    with open(os.path.join(data_dir, "customers.json"), "r") as f:
        customers = json.load(f)
        
    with open(os.path.join(data_dir, "vehicles.json"), "r") as f:
        vehicles = json.load(f)
        
    with open(os.path.join(data_dir, "products.json"), "r") as f:
        products = json.load(f)
    
    return {"customers": customers, "vehicles": vehicles, "products": products}

mock_data = load_mock_data()

# Define request models
class QuoteRequest(BaseModel):
    customer_id: str
    vehicle_info: Dict
    coverage_preferences: Optional[Dict] = None

class ChatMessage(BaseModel):
    customer_id: str
    message: str

# API endpoints
@app.get("/")
def read_root():
    return {"message": "Auto Insurance Quote API"}

@app.get("/customers")
def get_customers():
    return mock_data["customers"]

@app.get("/customers/{customer_id}")
def get_customer(customer_id: str):
    for customer in mock_data["customers"]:
        if customer["id"] == customer_id:
            return customer
    raise HTTPException(status_code=404, detail="Customer not found")

@app.get("/vehicles")
def get_vehicles():
    return mock_data["vehicles"]

@app.get("/products")
def get_products():
    return mock_data["products"]

@app.post("/quote")
def generate_quote(request: QuoteRequest):
    # This will be replaced with the actual agent system call
    # return process_quote_request(request.dict())
    
    # For now, return a mock response
    return {
        "quote_id": "q-12345",
        "customer_id": request.customer_id,
        "premium": 1250.00,
        "coverage": mock_data["products"][1],  # Standard coverage
        "vehicle": request.vehicle_info,
        "status": "approved"
    }

@app.post("/chat")
def process_chat(message: ChatMessage):
    # This will be replaced with the actual chat processing logic
    return {
        "response": f"Thank you for your message. We're processing your request for customer {message.customer_id}.",
        "extracted_info": {}
    }

if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=8000)


## Part 9: Enhancing the System with LLM Prompts

We've enhanced our agent system with sophisticated LLM prompts to better leverage Claude 3 Sonnet's capabilities. Let's examine some of these prompts and how they improve our system.

In [ ]:
# Let's look at the enhanced Risk Assessment Agent
!grep -A 30 "Risk Assessment Agent" backend/agent_system.py | head -30

The Risk Assessment Agent now uses a detailed prompt that instructs Claude to analyze customer and vehicle information to determine insurance risk levels. The prompt provides clear guidelines on what factors to consider and specifies the expected output format.

Similarly, we've enhanced the Coverage Determination Agent and Quote Generation Agent with sophisticated prompts.

In [ ]:
# Let's look at the enhanced Coverage Determination Agent
!grep -A 30 "Coverage Determination Agent" backend/agent_system.py | head -30

## Part 10: Adding a Chat-Based Interface

We've also created a chat agent that can process natural language requests for insurance quotes. This allows customers to get quotes through a conversational interface rather than filling out forms.

In [ ]:
# Let's look at the chat agent
!cat backend/chat_agent.py | head -30

Let's test the chat agent with a sample conversation:

In [ ]:
# Import our chat agent
import sys
sys.path.append('./backend')
from chat_agent import process_chat_message, reset_chat

In [ ]:
# Reset the chat to start fresh
reset_chat()

# Test with a sample message
response = process_chat_message("cust-001", "Hi, I'm looking for a quote for my 2020 Honda Accord.")
print("Customer: Hi, I'm looking for a quote for my 2020 Honda Accord.")
print(f"Assistant: {response['response']}")
print("\nExtracted Information:")
print(json.dumps(response['extracted_info'], indent=2))

In [ ]:
# Continue the conversation
response = process_chat_message("cust-001", "I'd like comprehensive coverage with a $500 deductible.")
print("Customer: I'd like comprehensive coverage with a $500 deductible.")
print(f"Assistant: {response['response']}")
print("\nExtracted Information:")
print(json.dumps(response['extracted_info'], indent=2))

## Next Steps

In the next part of this tutorial, we'll implement the frontend using React and Cloudscape components.